# MalariaCell MLOps — Model Evaluation Notebook

This notebook builds and evaluates the compact CNN used by the MalariaCell deployment.

**Dataset:** NIH Malaria Cell Images

Optimization techniques used:
- BatchNormalization + Dropout regularization
- Adam optimizer + ReduceLROnPlateau
- EarlyStopping
- Retraining with a frozen convolutional backbone and trainable classification head

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import MODEL_PATH, TEST_DIR, TRAIN_DIR
from src.model import build_malaria_cnn, configure_tf_memory, get_callbacks, save_model
from src.preprocessing import create_splits, dataset_counts, download_dataset
from src.retrain import make_tf_dataset_from_directory

import numpy as np
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns

configure_tf_memory()
print('ROOT', ROOT)
print('MODEL_PATH', MODEL_PATH)

## 1. Data acquisition

In [ ]:
source = download_dataset(force=False)
create_splits(source_dir=source, max_per_class=1500, rebuild=False)
print('Train counts:', dataset_counts(TRAIN_DIR))
print('Test counts:', dataset_counts(TEST_DIR))

## 2. Data processing

In [ ]:
train_ds = make_tf_dataset_from_directory(TRAIN_DIR, shuffle=True).cache().prefetch(1)
test_ds = make_tf_dataset_from_directory(TEST_DIR, shuffle=False).cache().prefetch(1)

for x, y in train_ds.take(1):
    print('Batch shape:', x.shape, 'labels:', y.numpy()[:8])
    plt.figure(figsize=(10, 3))
    for i in range(8):
        plt.subplot(1, 8, i + 1)
        plt.imshow(x[i].numpy())
        plt.axis('off')
        plt.title(int(y[i].numpy()))
    plt.show()

## 3. Model creation

In [ ]:
model = build_malaria_cnn()
model.summary()

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=8,
    callbacks=get_callbacks(MODEL_PATH),
)
save_model(model, MODEL_PATH)

## 4. Model testing — at least 4 metrics

In [ ]:
y_true, y_prob = [], []
for batch_x, batch_y in test_ds:
    probs = model.predict(batch_x, verbose=0).ravel()
    y_prob.extend(probs.tolist())
    y_true.extend(batch_y.numpy().ravel().tolist())

y_true = np.array(y_true)
y_prob = np.array(y_prob)
y_pred = (y_prob >= 0.5).astype(int)

metrics = {
    'accuracy': float(accuracy_score(y_true, y_pred)),
    'precision': float(precision_score(y_true, y_pred, zero_division=0)),
    'recall': float(recall_score(y_true, y_pred, zero_division=0)),
    'f1': float(f1_score(y_true, y_pred, zero_division=0)),
    'auc': float(roc_auc_score(y_true, y_prob)),
    'loss': float(history.history['loss'][-1]),
}
print(metrics)
print(classification_report(y_true, y_pred, target_names=['Parasitized', 'Uninfected']))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Parasitized', 'Uninfected'],
            yticklabels=['Parasitized', 'Uninfected'])
plt.title('Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.show()

fpr, tpr, _ = roc_curve(y_true, y_prob)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"AUC={metrics['auc']:.3f}")
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC Curve'); plt.legend()
plt.show()

hist = history.history
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(hist['loss'], label='train')
plt.plot(hist['val_loss'], label='val')
plt.title('Loss'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(hist['accuracy'], label='train')
plt.plot(hist['val_accuracy'], label='val')
plt.title('Accuracy'); plt.legend()
plt.show()

## 5. Deployment note

After training, save `models/malaria_cnn.keras` and deploy the API with Docker.
Retraining in the web UI freezes convolutional layers and fine-tunes the dense head on user-uploaded images stored in SQLite and `data/uploads/`.